# Churn data analysis

Here we load the data, see how it looks, and understand which customers leave more (churn) and why.

In [ ]:
# Where the data lives (one folder up from notebooks/)
import sys
from pathlib import Path
import pandas as pd

data_path = Path.cwd().parent / "data"
csv = list(data_path.glob("*.csv"))[0] if list(data_path.glob("*.csv")) else None
if csv is None:
    raise FileNotFoundError("Put a CSV file in the data/ folder")
df = pd.read_csv(csv)
df.head()

Check how many rows we have and if any column is missing values.

In [ ]:
df.shape
df.isnull().sum()

Clean up: drop rows with missing values and convert TotalCharges to numeric.

In [ ]:
df = df.dropna(how="any")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])
df.shape

How many customers stayed vs left (the thing we'll want to predict later).

In [ ]:
df["Churn"].value_counts()
df["Churn"].value_counts(normalize=True).round(3)

Plot that so we can see it clearly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df, x="Churn")
plt.title("Customers who stayed vs who left")
plt.show()

Check if customers with longer tenure leave less — often new customers are more likely to churn.

In [ ]:
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Months as customer by churn or not")
plt.show()

Same for what they pay per month: sometimes higher payers churn less (or the opposite, depending on the case).

In [ ]:
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly charge by churn or not")
plt.show()

Contract type: month-to-month customers usually churn more than one- or two-year contracts.

In [ ]:
pd.crosstab(df["Contract"], df["Churn"], normalize="index").round(3)

In [ ]:
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by contract type")
plt.xticks(rotation=15)
plt.show()

Cohorts: we group by tenure bands to see retention (how many stay in each group).

In [ ]:
df["tenure_band"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49+"])
retention = df.groupby("tenure_band", observed=True)["Churn"].apply(lambda x: (x == "No").mean()).round(3)
retention

In [ ]:
sns.barplot(x=retention.index.astype(str), y=retention.values)
plt.title("Proportion who stayed by tenure (months)")
plt.ylabel("Stayed")
plt.show()

Summary: which variables seem to move churn the most (so we know what to feed the model later).

In [ ]:
df["Churn_num"] = (df["Churn"] == "Yes").astype(int)
# Numeric columns only to see correlation with churn
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "Churn_num"]
df[numeric_cols].corr().round(3)